# Dataset Creation and Augmentation Pipeline

This notebook builds a structured dataset from raw hand landmark sequences previously recorded during the data acquisition stage. It loads and processes paired hand sequences, extracts spatial and motion-based features, and applies controlled data augmentation to improve model generalization. The resulting dataset is formatted as numerical tensors suitable for machine learning workflows and saved in a compressed format for efficient storage and training.

In [5]:
import os
import pickle
import numpy as np
import json
from typing import Dict, Tuple, List
from scipy.interpolate import interp1d
from landmarkers.inferences import InferenceSequence
from landmarkers.landmarks import LandmarksSequence


with open('../config.json', 'r') as f:
    config = json.load(f)

common_config = config['common']
ACTIONS = common_config['actions']
SEQUENCE_LENGTH = common_config['sequence_length']
DATA_PATH = common_config['data_path']

dataset_config = config['create_dataset']
OUTPUT_PATH = dataset_config['output_path']
augmentation_counts: Dict[str, int] = dataset_config['augmentation_counts']
aug_config = dataset_config['augmentation']
NOISE_STD = aug_config['noise_std']
SCALE_RANGE = tuple(aug_config['scale_range'])
DROPOUT_PROB = aug_config['dropout_prob']
TEMPORAL_PROB = aug_config['temporal_prob']


def add_spatial_noise(seq: np.ndarray, std: float) -> np.ndarray:
    return seq + np.random.normal(0, std, seq.shape)


def scale_sequence(seq: np.ndarray, scale: float) -> np.ndarray:
    return seq * scale


def frame_dropout(seq: np.ndarray, idx: int) -> np.ndarray:
    seq = np.array(seq)
    seq[idx] = 0
    return seq


def temporal_interpolation(
    seq: np.ndarray,
    sub_start: int,
    sub_end: int,
    sequence_length: int
) -> np.ndarray:
    sub_seq = seq[sub_start:sub_end]
    x_old = np.linspace(0, 1, num=len(sub_seq))
    x_new = np.linspace(0, 1, num=sequence_length)
    f = interp1d(x_old, sub_seq, axis=0)
    return f(x_new)


def temporal_padding(
    seq: np.ndarray,
    sub_start: int,
    sub_end: int,
    sequence_length: int
) -> np.ndarray:
    sub_seq = seq[sub_start:sub_end]
    pad_len = sequence_length - len(sub_seq)
    pad_front = pad_len // 2
    pad_back = pad_len - pad_front
    return np.vstack([
        np.tile(sub_seq[0], (pad_front, 1, 1)),
        sub_seq,
        np.tile(sub_seq[-1], (pad_back, 1, 1))
    ])


def augment_sequence(
    seq: np.ndarray,
    sequence_length: int,
    noise_std: float = NOISE_STD,
    scale_range: Tuple[float, float] = SCALE_RANGE,
    dropout_prob: float = DROPOUT_PROB,
    temporal_prob: float = TEMPORAL_PROB
) -> np.ndarray:
    seq = np.array(seq)
    orig_len = len(seq)

    if np.random.rand() < 1.0:
        seq = add_spatial_noise(seq, noise_std)

    if np.random.rand() < 1.0:
        scale = np.random.uniform(*scale_range)
        seq = scale_sequence(seq, scale)

    if np.random.rand() < dropout_prob:
        idx = np.random.randint(0, orig_len)
        seq = frame_dropout(seq, idx)

    if np.random.rand() < temporal_prob:
        sub_start = np.random.randint(0, orig_len // 2)
        sub_end = sub_start + np.random.randint(orig_len // 2, orig_len)
        sub_end = min(sub_end, orig_len)

        if np.random.rand() < 0.5:
            seq = temporal_interpolation(seq, sub_start, sub_end, sequence_length)
        else:
            seq = temporal_padding(seq, sub_start, sub_end, sequence_length)

    return seq.astype(np.float32)


def sigmoid(x):
    return np.where(
        x >= 0,
        1 / (1 + np.exp(-x)),
        np.exp(x) / (1 + np.exp(x))
    )


def get_centroid_velocity_norm_sequence(landmark_sequence) -> np.ndarray:
    time_stamps = np.array(landmark_sequence.time_stamps_ms, dtype=np.float32)
    centroids = np.array(landmark_sequence.centroid(), dtype=np.float32)

    d_centroids = np.diff(centroids, axis=0)
    d_time = np.diff(time_stamps)[:, None]
    d_time[d_time == 0] = 1e-6

    velocity = d_centroids / d_time
    norms = np.linalg.norm(velocity, axis=1, keepdims=True)
    norms[norms == 0] = 1e-6

    norm_velocity = (sigmoid(norms) / norms) * velocity
    norm_velocity = np.vstack([
        np.zeros((1, 3), dtype=np.float32),
        norm_velocity
    ])

    return norm_velocity.astype(np.float32)


def load_sequence_from_pkl(pkl_path: str) -> np.ndarray:
    with open(pkl_path, "rb") as f:
        inference_sequence: InferenceSequence = pickle.load(f)

    landmark_sequence: LandmarksSequence = inference_sequence.landmarks_sequence
    landmark_sequence = landmark_sequence.resample(target_frames=20)

    velocity = get_centroid_velocity_norm_sequence(landmark_sequence)

    landmark_sequence = landmark_sequence.centered(0)
    landmarks_array = landmark_sequence.array

    frames = np.concatenate(
        [landmarks_array, velocity[:, None, :]],
        axis=1
    )

    return np.array(frames, dtype=np.float32)


def combine_hands(seq_right: np.ndarray, seq_left: np.ndarray) -> np.ndarray:
    return np.concatenate([seq_right, seq_left], axis=1)


def build_dataset() -> Tuple[np.ndarray, np.ndarray]:
    label_map: Dict[str, int] = {label: i for i, label in enumerate(ACTIONS)}

    sequences: List[np.ndarray] = []
    labels: List[int] = []

    for action in ACTIONS:
        action_path = os.path.join(DATA_PATH, action)
        if not os.path.isdir(action_path):
            continue

        for seq_folder in os.listdir(action_path):
            seq_path = os.path.join(action_path, seq_folder)
            if not os.path.isdir(seq_path):
                continue

            right_pkl = os.path.join(seq_path, "right.pkl")
            left_pkl = os.path.join(seq_path, "left.pkl")

            if not (os.path.exists(right_pkl) and os.path.exists(left_pkl)):
                continue

            seq_right = load_sequence_from_pkl(right_pkl)
            seq_left = load_sequence_from_pkl(left_pkl)

            window = combine_hands(seq_right, seq_left)

            sequences.append(window)
            labels.append(label_map[action])

            n_aug = augmentation_counts.get(action, 1)
            for _ in range(n_aug):
                aug = augment_sequence(
                    window,
                    sequence_length=SEQUENCE_LENGTH,
                    temporal_prob=0
                )
                sequences.append(aug)
                labels.append(label_map[action])

    X = np.stack(sequences)
    y = np.array(labels, dtype=np.int32)

    return X, y


def save_dataset(X: np.ndarray, y: np.ndarray):
    np.savez_compressed(
        OUTPUT_PATH,
        X=X,
        y=y
    )
    print("X.shape =", X.shape)
    print("y.shape =", y.shape)


def main():
    X, y = build_dataset()
    save_dataset(X, y)


if __name__ == "__main__":
    main()

/tmp/ipykernel_65331/2812919147.py:112: RuntimeWarning: overflow encountered in exp
  np.exp(x) / (1 + np.exp(x))
/tmp/ipykernel_65331/2812919147.py:112: RuntimeWarning: invalid value encountered in divide
  np.exp(x) / (1 + np.exp(x))


X.shape = (26260, 20, 44, 3)
y.shape = (26260,)
